# Diabetes RAG – Ingestion, Chunking, Embeddings & Retrieval

Loads the two diabetes guideline PDFs from GitHub, cleans and organizes them by section, creates page-aware chunks, stores embeddings in Chroma, and tests retrieval.

## 1. Download the PDFs from GitHub

In [1]:
!git clone https://github.com/Nourmohamed904/Diabetes_RAG_Hackathon.git

Cloning into 'Diabetes_RAG_Hackathon'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 75 (delta 26), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 883.26 KiB | 3.18 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [2]:
import os

PDF_FOLDER = "/content/Diabetes_RAG_Hackathon/data"

if not os.path.isdir(PDF_FOLDER):
    raise FileNotFoundError(
        f"PDF folder not found: {PDF_FOLDER}\n"
        "Check the repository structure or update PDF_FOLDER."
    )

PDF_PATHS = sorted(
    os.path.join(PDF_FOLDER, file)
    for file in os.listdir(PDF_FOLDER)
    if file.lower().endswith(".pdf")
)

print("Number of PDFs:", len(PDF_PATHS))
for pdf in PDF_PATHS:
    print("-", os.path.basename(pdf))

Number of PDFs: 2
- Type-1 diabetes.pdf
- Type-2 diabetes.pdf


## 2. Install and import the required libraries

In [3]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-chroma chromadb fastembed pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0

In [4]:
import os
import re
import shutil
from collections import Counter

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

/tmp/ipykernel_2014/1583142254.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 3. Load the PDFs and add metadata

In [5]:
all_pages = []

for pdf_path in PDF_PATHS:
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    document_name = os.path.basename(pdf_path)

    for page in pages:
        page.metadata["document_name"] = document_name
        page.metadata["page_number"] = page.metadata.get("page", 0) + 1

    all_pages.extend(pages)

print("Total PDFs:", len(PDF_PATHS))
print("Total pages:", len(all_pages))

Total PDFs: 2
Total pages: 195


## 4. Clean the extracted text

In [6]:
def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

for page in all_pages:
    page.page_content = clean_text(page.page_content)

print("Cleaning completed.")

Cleaning completed.


### 4b. Remove repeated boilerplate (copyright footer / page-of-64 lines)

Every page repeats the same NICE copyright notice and a "Page X of NN" footer, and often the guideline title as a running header. This text carries no medical meaning, but it gets embedded into every chunk that touches a page boundary, diluting the chunk's embedding and pushing genuinely relevant chunks further down the similarity ranking. Stripping it before chunking keeps each chunk focused on actual guideline content.

In [7]:
# ==========================================
# 5. Remove repeated boilerplate
# ==========================================

# FIX: the previous version split the copyright notice into three separately-
# optional pieces, so it only ever matched "© NICE 2026. All rights reserved."
# and left the rest of the sentence ("Subject to Notice of rights (URL).")
# behind on every single page. This version uses one DOTALL, non-greedy match
# from "All rights reserved." through the closing "notice-of-rights).", so the
# whole clause is removed regardless of the exact wording in between.
BOILERPLATE_PATTERNS = [
    # NICE copyright/footer (full clause, not just the first sentence)
    r"©\s*NICE\s*\d{4}\.\s*All rights reserved\..*?notice-of-rights\)\.?",

    # Page X of Y
    r"Page\s+\d+\s+of\s+\d+",

    # Running headers
    r"Type\s+1\s+diabetes\s+in\s+adults:\s*"
    r"diagnosis\s+and\s+management\s*\(NG17\)",

    r"Type\s+2\s+diabetes\s+in\s+adults:\s*"
    r"management\s*\(NG28\)",
]


# Compile each pattern separately.
# DOTALL so the copyright clause can be matched across the newline that PDF
# extraction leaves before the URL.
_BOILERPLATE_REGEXES = [
    re.compile(pattern, flags=re.IGNORECASE | re.DOTALL)
    for pattern in BOILERPLATE_PATTERNS
]


def remove_boilerplate(text):
    """
    Remove repeated PDF headers/footers without accidentally
    deleting large parts of the medical content.
    """

    for pattern in _BOILERPLATE_REGEXES:
        text = pattern.sub(" ", text)

    # Clean spaces left after removal
    text = re.sub(r"[ \t]+", " ", text)

    # Clean excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# ------------------------------------------
# Test before / after
# ------------------------------------------

sample_index = min(16, len(all_pages) - 1)

sample_before = all_pages[sample_index].page_content

sample_after = remove_boilerplate(sample_before)

print("===== BEFORE boilerplate removal =====")
print(sample_before[-500:])

print("\n===== AFTER boilerplate removal =====")
print(sample_after[-500:])

assert "All rights reserved" not in sample_after, "Copyright clause was not fully removed"
assert "notice-of-rights" not in sample_after.lower(), "Copyright clause was not fully removed"


# ------------------------------------------
# Apply to all pages
# ------------------------------------------

for page in all_pages:
    page.page_content = remove_boilerplate(page.page_content)

print(
    "\nBoilerplate removal applied to all",
    len(all_pages),
    "pages"
)

===== BEFORE boilerplate removal =====
able at consultations. Follow the principles on 
communication in NICE's guideline on patient experience in adult NHS services. 
[2015] 
1.6.5 If HbA1c monitoring is invalid because of disturbed erythrocyte turnover or 
abnormal haemoglobin type, estimate trends in blood glucose control using 1 of 
Type 1 diabetes in adults: diagnosis and management (NG17)
© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and-
conditions#notice-of-rights).
Page 17 of
64

===== AFTER boilerplate removal =====
International Federation of 
Clinical Chemistry (IFCC) standardisation. [2015] 
1.6.4 Tell adults with type 1 diabetes their HbA1c results after each measurement and 
have their most recent result available at consultations. Follow the principles on 
communication in NICE's guideline on patient experience in adult NHS services. 
[2015] 
1.6.5 If HbA1c monitoring is invalid because of disturbed erythrocyte turnover or 

In [8]:
print("Document:", all_pages[0].metadata["document_name"])
print("Page:", all_pages[0].metadata["page_number"])
print("\nSample text:\n")
print(all_pages[0].page_content[:1500])

Document: Type-1 diabetes.pdf
Page: 1

Sample text:

Type 1 diabetes in adults: 
diagnosis and management 
NICE guideline 
Published: 26 August 2015 
Last updated: 17 August 2022 
www.nice.org.uk/guidance/ng17


## 5. Detect and clean section headings

In [9]:
def is_section_heading(line):
    line = line.strip()
    if not line:
        return False

    # Main sections such as 1.1, 1.2, 1.10
    pattern = r"^\d+\.\d+\s+.+"
    if not re.match(pattern, line):
        return False

    # FIX: exclude Table-of-Contents lines. They match the same "X.Y title" shape
    # but end with dot leaders + a page number, e.g.:
    # "1.1 Diagnosis and early care plan ..................... 6"
    # Without this check, get_first_content_page() below detects the Contents
    # page itself as the start of real content instead of the actual first
    # recommendations page.
    if re.search(r"\.{3,}\s*\d+\s*$", line):
        return False

    return True


def clean_section_heading(line):
    line = line.strip()

    # Example:
    # 1.1 Diagnosis and early care plan ............ 6
    # -> 1.1 Diagnosis and early care plan
    line = re.sub(r"\s*\.{3,}\s*\d+\s*$", "", line)

    return line.strip()

In [10]:
test_lines = [
    "1.1 Diagnosis and early care plan",
    "1.2 Support and individualised care",
    "1.10 Ketone monitoring and managing diabetic ketoacidosis",
    "1.1.1 Make an initial diagnosis of type 1 diabetes",
    "People with diabetes should...",
    "1.1 Diagnosis and early care plan ......................................................................................................... 6",  # ToC line, should be False
]

for line in test_lines:
    print(is_section_heading(line), "→", line)

True → 1.1 Diagnosis and early care plan
True → 1.2 Support and individualised care
True → 1.10 Ketone monitoring and managing diabetic ketoacidosis
False → 1.1.1 Make an initial diagnosis of type 1 diabetes
False → People with diabetes should...
False → 1.1 Diagnosis and early care plan ......................................................................................................... 6


## 6. Group pages by section

In [11]:
def get_first_content_page(pages):
    """
    Find the first real guideline section,
    while ignoring section names appearing in the table of contents.
    """
    for page in pages:
        for line in page.page_content.splitlines():
            line = line.strip()
            if not line:
                continue
            # Table-of-contents lines usually contain dotted leaders:
            # 1.1 Diagnosis ............ 6
            if "..." in line:
                continue
            # Real main-section heading:
            # 1.1 Diagnosis and early care plan
            if is_section_heading(line):
                return page.metadata["page_number"]
    return 1


def group_pages_by_section(pages):

    grouped_sections = []

    # Group pages by PDF/document
    pages_by_document = {}

    for page in pages:
        document_name = page.metadata["document_name"]

        pages_by_document.setdefault(
            document_name, []
        ).append(page)

    # Process each PDF independently
    for document_name, document_pages in pages_by_document.items():

        current_section = None
        current_pages = []

        # Detect first content page automatically
        first_content_page = get_first_content_page(
            document_pages
        )

        print(
            f"{document_name} -> "
            f"first content page: {first_content_page}"
        )

        for page in document_pages:

            page_number = page.metadata["page_number"]

            if page_number < first_content_page:
                continue

            for raw_line in page.page_content.splitlines():

                line = raw_line.strip()

                if not line:
                    continue

                # New section
                if is_section_heading(line):

                    if current_section is not None:
                        grouped_sections.append({
                            "document_name": document_name,
                            "section": current_section,
                            "pages": current_pages.copy()
                        })

                    current_section = clean_section_heading(line)

                    current_pages = [{
                        "page_number": page_number,
                        "text": ""
                    }]

                # Normal content
                elif current_section is not None:

                    if (
                        not current_pages
                        or current_pages[-1]["page_number"] != page_number
                    ):
                        current_pages.append({
                            "page_number": page_number,
                            "text": line
                        })

                    else:
                        if current_pages[-1]["text"]:
                            current_pages[-1]["text"] += "\n" + line
                        else:
                            current_pages[-1]["text"] = line

        # Save final section
        if current_section is not None:
            grouped_sections.append({
                "document_name": document_name,
                "section": current_section,
                "pages": current_pages.copy()
            })

    return grouped_sections


# Run
grouped_sections = group_pages_by_section(all_pages)

print("Number of grouped sections:", len(grouped_sections))

if not grouped_sections:
    raise ValueError(
        "No sections were detected. "
        "Check section-heading detection and PDF extraction."
    )

first_section = grouped_sections[0]

print("\n===== FIRST SECTION =====")
print("Document:", first_section["document_name"])
print("Section:", first_section["section"])
print("Number of pages:", len(first_section["pages"]))

print(
    "Pages:",
    [p["page_number"] for p in first_section["pages"]]
)

print("\nFirst page text:")
print(first_section["pages"][0]["text"][:1000])

Type-1 diabetes.pdf -> first content page: 6
Type-2 diabetes.pdf -> first content page: 7
Number of grouped sections: 59

===== FIRST SECTION =====
Document: Type-1 diabetes.pdf
Section: 1.1 Diagnosis and early care plan
Number of pages: 5
Pages: [6, 7, 8, 9, 10]

First page text:
Initial diagnosis
1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults
presenting with hyperglycaemia. Bear in mind that people with type 1 diabetes
typically (but not always) have 1 or more of:
• ketosis
• rapid weight loss
• age of onset under 50 years
• body mass index (BMI) below 25 kg/m2


**Chunk size note:** originally 850/150. Lowered to **300/60** after diagnosing Q5 ("insulin plan" question): at 850 chars, recommendation 1.7.1 (insulin regimen) was sharing a chunk with the unrelated "Levemir discontinuation" notice right after it, diluting the chunk's embedding enough that it dropped out of the top-6 results. At 300 chars, 1.7.1 gets an isolated chunk. Verified this doesn't break the other working recommendations (1.6.6 HbA1c target, 1.13.1 metformin first-line) — both still land cleanly in their own chunks at this size.

## 7. Create page-aware chunks

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", " ", ""],
)


def create_all_chunks(grouped_sections, splitter):
    all_chunks = []
    chunk_counter = 1

    for section in grouped_sections:
        section_text = ""
        page_boundaries = []

        for page in section["pages"]:
            start_position = len(section_text)
            section_text += page["text"] + "\n"
            end_position = len(section_text)

            page_boundaries.append({
                "page_number": page["page_number"],
                "start": start_position,
                "end": end_position,
            })

        if not page_boundaries:
            continue

        section_chunks = splitter.split_text(section_text)
        search_start = 0

        for chunk_text in section_chunks:
            chunk_start = section_text.find(chunk_text, search_start)
            if chunk_start == -1:
                chunk_start = search_start

            chunk_end = chunk_start + len(chunk_text)

            chunk_pages = [
                boundary["page_number"]
                for boundary in page_boundaries
                if boundary["end"] > chunk_start
                and boundary["start"] < chunk_end
            ]

            if not chunk_pages:
                chunk_pages = [page_boundaries[0]["page_number"]]

            chunk = Document(
                page_content=chunk_text,
                metadata={
                    "document_name": section["document_name"],
                    "section": section["section"],
                    "page_number": chunk_pages[0],
                    "page_numbers": chunk_pages,
                    "chunk_id": f"chunk_{chunk_counter:04d}",
                },
            )

            all_chunks.append(chunk)
            chunk_counter += 1
            search_start = chunk_start + 1

    return all_chunks

In [13]:
chunks = create_all_chunks(grouped_sections, splitter)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3], 1):
    print("=" * 80)
    print(f"CHUNK {i}")
    print("Text:", chunk.page_content[:500])
    print("Metadata:", chunk.metadata)

Total chunks: 1058
CHUNK 1
Text: Initial diagnosis
1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults
presenting with hyperglycaemia. Bear in mind that people with type 1 diabetes
typically (but not always) have 1 or more of:
• ketosis
• rapid weight loss
• age of onset under 50 years
Metadata: {'document_name': 'Type-1 diabetes.pdf', 'section': '1.1 Diagnosis and early care plan', 'page_number': 6, 'page_numbers': [6], 'chunk_id': 'chunk_0001'}
CHUNK 2
Text: • ketosis
• rapid weight loss
• age of onset under 50 years
• body mass index (BMI) below 25 kg/m2
• personal and/or family history of autoimmune disease. [2015, amended
2022]
1.1.2 Do not use age or BMI alone to exclude or diagnose type 1 diabetes in adults.
[2022]
Metadata: {'document_name': 'Type-1 diabetes.pdf', 'section': '1.1 Diagnosis and early care plan', 'page_number': 6, 'page_numbers': [6, 7], 'chunk_id': 'chunk_0002'}
CHUNK 3
Text: [2022]
1.1.3 Take into consideration the possibility of ot

## 8. Validate chunk metadata

In [14]:
required_fields = [
    "document_name",
    "section",
    "page_number",
    "page_numbers",
    "chunk_id",
]

for i, chunk in enumerate(chunks):
    for field in required_fields:
        assert field in chunk.metadata, f"Missing '{field}' in chunk {i}"

chunk_ids = [chunk.metadata["chunk_id"] for chunk in chunks]
assert len(chunk_ids) == len(set(chunk_ids)), "Duplicate chunk IDs found"

print("Metadata validation passed.")
print("Unique chunks:", len(chunks))

document_counts = Counter(
    chunk.metadata["document_name"] for chunk in chunks
)

print("\nChunks per document:")
for document, count in document_counts.items():
    print("-", document, ":", count)

Metadata validation passed.
Unique chunks: 1058

Chunks per document:
- Type-1 diabetes.pdf : 356
- Type-2 diabetes.pdf : 702


## 9. Generate embeddings and create the Chroma vector store

In [15]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = FastEmbedEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

# Chroma metadata must use scalar values, so convert page_numbers to a string.
cleaned_chunks = []

for doc in chunks:
    clean_meta = {}

    for key, value in doc.metadata.items():
        if value is None:
            clean_meta[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            clean_meta[key] = value
        else:
            clean_meta[key] = str(value)

    doc.metadata = clean_meta
    cleaned_chunks.append(doc)

persist_directory = "./chroma_db"

if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

vector_db = Chroma.from_documents(
    documents=cleaned_chunks,
    embedding=embedding_model,
    persist_directory=persist_directory,
    collection_name="diabetes_educational_rag",
)

stored_count = vector_db._collection.count()

print("Input chunks :", len(chunks))
print("Stored vectors:", stored_count)

assert stored_count == len(chunks), "Chroma count does not match input chunks."
print("Chroma indexing completed successfully.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Input chunks : 1058
Stored vectors: 1058
Chroma indexing completed successfully.


## 10. Retrieval test

In [16]:
def _extract_pages(chunk_pages, fallback_page):
    if isinstance(chunk_pages, list):
        return [
            int(p)
            for p in chunk_pages
            if str(p).strip().lstrip("-").isdigit()
        ]

    if isinstance(chunk_pages, str):
        found = re.findall(r"\d+", chunk_pages)
        if found:
            return [int(p) for p in found]

    return [fallback_page] if fallback_page is not None else []


def retrieve_with_similarity(question, k=4):
    return vector_db.similarity_search_with_relevance_scores(question, k=k)


def print_retrieval_results(question, k=4):
    results = retrieve_with_similarity(question, k=k)

    print(f"QUESTION: {question}\n")

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"Rank {rank}")
        print("Document :", doc.metadata.get("document_name"))
        print("Page     :", doc.metadata.get("page_number"))
        print("Pages    :", doc.metadata.get("page_numbers"))
        print("Section  :", doc.metadata.get("section"))
        print("Chunk ID :", doc.metadata.get("chunk_id"))
        print("Score    :", round(score, 4))
        print("Text     :", doc.page_content[:500].replace("\n", " "), "...")
        print()

    return results

In [17]:
results = print_retrieval_results(
    "What are the diagnostic criteria for diabetes?",
    k=4,
)

QUESTION: What are the diagnostic criteria for diabetes?

Rank 1
Document : Type-1 diabetes.pdf
Page     : 6
Pages    : [6]
Section  : 1.1 Diagnosis and early care plan
Chunk ID : chunk_0001
Score    : 0.6632
Text     : Initial diagnosis 1.1.1 Make an initial diagnosis of type 1 diabetes on clinical grounds in adults presenting with hyperglycaemia. Bear in mind that people with type 1 diabetes typically (but not always) have 1 or more of: • ketosis • rapid weight loss • age of onset under 50 years ...

Rank 2
Document : Type-1 diabetes.pdf
Page     : 6
Pages    : [6, 7]
Section  : 1.1 Diagnosis and early care plan
Chunk ID : chunk_0002
Score    : 0.6281
Text     : • ketosis • rapid weight loss • age of onset under 50 years • body mass index (BMI) below 25 kg/m2 • personal and/or family history of autoimmune disease. [2015, amended 2022] 1.1.2 Do not use age or BMI alone to exclude or diagnose type 1 diabetes in adults. [2022] ...

Rank 3
Document : Type-1 diabetes.pdf
Page     : 8
Page

**Note:** the test questions are now hardcoded directly in this notebook instead of parsed
from `RAG_Test_Questions.pdf`. Parsing a PDF table with regex is fragile — it silently breaks
(and produces `0/0` results) whenever the PDF's layout changes. Editing questions here is
also easier: just edit the list below.


## 11b. Load the evaluation dataset from Excel

Replaces the hardcoded question list with `Diabetes_RAG_Evaluation_Dataset.xlsx` (20 questions across 4 categories: Supported, Difficult Type 1 vs Type 2, Ambiguous, Out of scope). Two ways to get the file in:

- **Option A (recommended):** commit the .xlsx into the GitHub repo (e.g. under `eval/`), then it's already available at `EXCEL_PATH` below after the `git clone` step.
- **Option B (quick, Colab only):** run the upload cell first, pick the file from your computer, then run the loading cell.

In [18]:
import pandas as pd

# Option A path (uncomment/adjust if the file lives in the cloned repo instead of an upload)
EXCEL_PATH = "/content/Diabetes_RAG_Hackathon/Diabetes_RAG_Evaluation/Diabetes_RAG_Evaluation.xlsx"

eval_df = pd.read_excel(EXCEL_PATH, sheet_name="Evaluation Dataset")
print("Loaded", len(eval_df), "evaluation questions")
eval_df.head()

Loaded 20 evaluation questions


,#,Question,Category,Diabetes Type,Expected Topic,Expected Guideline,Expected Page(s),Clinical Validation Note
0,1,What is the target HbA1c for adults with type ...,Supported,Type 1,HbA1c target,NG17,18,NG17 rec 1.6.6 states an explicit numeric targ...
1,2,What is the preferred insulin regimen for adul...,Supported,Type 1,Insulin regimen choice,NG17,24,NG17 rec 1.7.1-1.7.2: multiple daily injection...
2,3,What clinical signs suggest someone might have...,Supported,Type 1,Diagnostic signs,NG17,6,NG17 rec 1.1.1 lists 5 specific signs (ketosis...
3,4,What should be done for someone with type 1 di...,Supported,Type 1,Impaired hypoglycaemia awareness management,NG17,30,NG17 recs 1.9.1-1.9.8: annual assessment (Gold...
4,5,What is usually the first medicine given for t...,Supported,Type 2,First-line pharmacological treatment,NG28,32,NG28 rec 1.13.1: metformin (modified-release i...


In [19]:
# Map "Expected Guideline" text to the document_name prefix used elsewhere in this notebook
GUIDELINE_TO_TAG = {
    "NG17": "type-1",
    "NG28": "type-2",
}


def parse_guideline_tags(text):
    """'NG17' -> ['type-1']; 'NG17 + NG28' -> ['type-1','type-2']; 'None' -> []"""
    if not isinstance(text, str):
        return []
    return [
        GUIDELINE_TO_TAG[tag]
        for tag in GUIDELINE_TO_TAG
        if tag in text
    ]


def parse_first_page(text):
    """'18' -> 18; '17 (NG17), 12 (NG28)' -> 17; 'N/A' -> None"""
    if not isinstance(text, str):
        return None
    import re
    match = re.search(r"\d+", text)
    return int(match.group()) if match else None


eval_questions = []
for _, row in eval_df.iterrows():
    eval_questions.append({
        "number": row["#"],
        "question": row["Question"],
        "category": row["Category"],
        "diabetes_type": row["Diabetes Type"],
        "expected_topic": row["Expected Topic"],
        "expected_guidelines": parse_guideline_tags(row["Expected Guideline"]),
        "expected_page": parse_first_page(row["Expected Page(s)"]),
        "clinical_note": row["Clinical Validation Note"],
    })

# Split by category so you can test each group separately, same idea as
# supported_questions / unsupported_questions used earlier in this notebook.
by_category = {}
for item in eval_questions:
    by_category.setdefault(item["category"], []).append(item)

for category, items in by_category.items():
    print(f"{category}: {len(items)} questions")

Supported: 9 questions
Difficult Type 1 vs Type 2: 4 questions
Ambiguous: 4 questions
Out of scope: 3 questions


In [20]:
def check_match(
    results,
    expected_guidelines,
    expected_page,
    page_tolerance=3
):
    """
    Evaluate retrieval against the Excel evaluation dataset.

    A retrieval is considered a full match when at least one retrieved
    chunk:
      1. Comes from one of the expected guidelines.
      2. Contains the expected page (within page_tolerance).

    Returns:
        source_match
        page_match
        match
    """

    expected_guidelines = expected_guidelines or []

    source_match = False
    page_match = False
    full_match = False

    for doc, _score in results:

        doc_name = doc.metadata.get(
            "document_name",
            ""
        ).lower()

        # ---------------------------------
        # Source / guideline match
        # ---------------------------------

        current_source_match = any(
            guideline.lower() in doc_name
            for guideline in expected_guidelines
        )

        if not current_source_match:
            continue

        source_match = True

        # ---------------------------------
        # Page match
        # ---------------------------------

        chunk_pages = _extract_pages(
            doc.metadata.get("page_numbers"),
            doc.metadata.get("page_number")
        )

        # If Excel contains an expected page
        if expected_page is not None:

            current_page_match = any(
                abs(page - expected_page) <= page_tolerance
                for page in chunk_pages
            )

        else:
            current_page_match = True

        if current_page_match:
            page_match = True
            full_match = True
            break

    return {
        "source_match": source_match,
        "page_match": page_match,
        "match": full_match,
    }

In [21]:
print("===== EXCEL RETRIEVAL EVALUATION =====\n")

evaluation_results = []

for item in eval_questions:

    results = retrieve_with_similarity(
        item["question"],
        k=4
    )

    outcome = check_match(
        results=results,
        expected_guidelines=item["expected_guidelines"],
        expected_page=item["expected_page"],
        page_tolerance=3
    )

    evaluation_results.append({
        "number": item["number"],
        "question": item["question"],
        "category": item["category"],
        "expected_guidelines": item["expected_guidelines"],
        "expected_page": item["expected_page"],
        "source_match": outcome["source_match"],
        "page_match": outcome["page_match"],
        "match": outcome["match"],
    })

    print(f"Q{item['number']}: {item['question']}")
    print("Expected guidelines:", item["expected_guidelines"])
    print("Expected page:", item["expected_page"])
    print("Source match:", outcome["source_match"])
    print("Page match:", outcome["page_match"])
    print("FULL MATCH:", outcome["match"])
    print("-" * 80)


===== EXCEL RETRIEVAL EVALUATION =====

Q1: What is the target HbA1c for adults with type 1 diabetes?
Expected guidelines: ['type-1']
Expected page: 18
Source match: True
Page match: True
FULL MATCH: True
--------------------------------------------------------------------------------
Q2: What is the preferred insulin regimen for adults newly diagnosed with type 1 diabetes?
Expected guidelines: ['type-1']
Expected page: 24
Source match: True
Page match: True
FULL MATCH: True
--------------------------------------------------------------------------------
Q3: What clinical signs suggest someone might have type 1 diabetes?
Expected guidelines: ['type-1']
Expected page: 6
Source match: True
Page match: True
FULL MATCH: True
--------------------------------------------------------------------------------
Q4: What should be done for someone with type 1 diabetes who no longer feels their hypoglycaemia symptoms?
Expected guidelines: ['type-1']
Expected page: 30
Source match: True
Page match: 

In [22]:
evaluation_df = pd.DataFrame(evaluation_results)

total_questions = len(evaluation_df)

source_matches = evaluation_df["source_match"].sum()
page_matches = evaluation_df["page_match"].sum()
full_matches = evaluation_df["match"].sum()

print("\n===== EVALUATION SUMMARY =====")

print("Total questions :", total_questions)

print(
    "Source matches  :",
    source_matches,
    f"({source_matches / total_questions:.2%})"
)

print(
    "Page matches    :",
    page_matches,
    f"({page_matches / total_questions:.2%})"
)

print(
    "Full matches    :",
    full_matches,
    f"({full_matches / total_questions:.2%})"
)


===== EVALUATION SUMMARY =====
Total questions : 20
Source matches  : 16 (80.00%)
Page matches    : 15 (75.00%)
Full matches    : 15 (75.00%)


In [23]:
def diagnose_page_retrieval(item, k_values=(4, 6)):

    expected_guidelines = item["expected_guidelines"]
    expected_page = item["expected_page"]

    print(f"Q{item['number']}: {item['question']}")
    print(f"Expected guidelines: {expected_guidelines}")
    print(f"Expected page: {expected_page}")

    for k in k_values:

        results = retrieve_with_similarity(
            item["question"],
            k=k
        )

        found_pages = []
        expected_page_present = False

        for doc, score in results:

            doc_name = doc.metadata.get(
                "document_name",
                ""
            ).lower()

            # Check expected guideline
            source_match = any(
                guideline.lower() in doc_name
                for guideline in expected_guidelines
            )

            if not source_match:
                continue

            chunk_pages = _extract_pages(
                doc.metadata.get("page_numbers"),
                doc.metadata.get("page_number")
            )

            found_pages.extend(chunk_pages)

            if (
                expected_page is not None
                and any(
                    abs(page - expected_page) <= 3
                    for page in chunk_pages
                )
            ):
                expected_page_present = True

        status = (
            "FOUND in top-k"
            if expected_page_present
            else "MISSING from top-k"
        )

        print(
            f"  k={k}: "
            f"pages retrieved = {sorted(set(found_pages))} "
            f"-> {status}"
        )

    print("=" * 80)

In [24]:
print(
    "===== PAGE-RETRIEVAL DIAGNOSTIC "
    "(k=4 vs k=6) =====\n"
)

for item in eval_questions:
    diagnose_page_retrieval(item)

===== PAGE-RETRIEVAL DIAGNOSTIC (k=4 vs k=6) =====

Q1: What is the target HbA1c for adults with type 1 diabetes?
Expected guidelines: ['type-1']
Expected page: 18
  k=4: pages retrieved = [18, 29] -> FOUND in top-k
  k=6: pages retrieved = [17, 18, 23, 29] -> FOUND in top-k
Q2: What is the preferred insulin regimen for adults newly diagnosed with type 1 diabetes?
Expected guidelines: ['type-1']
Expected page: 24
  k=4: pages retrieved = [24, 26, 27, 31] -> FOUND in top-k
  k=6: pages retrieved = [24, 26, 27, 31, 39] -> FOUND in top-k
Q3: What clinical signs suggest someone might have type 1 diabetes?
Expected guidelines: ['type-1']
Expected page: 6
  k=4: pages retrieved = [6, 7, 30, 49] -> FOUND in top-k
  k=6: pages retrieved = [6, 7, 30, 46, 49, 51] -> FOUND in top-k
Q4: What should be done for someone with type 1 diabetes who no longer feels their hypoglycaemia symptoms?
Expected guidelines: ['type-1']
Expected page: 30
  k=4: pages retrieved = [29, 31] -> FOUND in top-k
  k=6: pa

In [25]:
# ==========================================
# Diagnose Q5
# ==========================================

q5 = eval_questions[4]

print("Question:", q5["question"])
print("Expected guideline:", q5["expected_guidelines"])
print("Expected page:", q5["expected_page"])
print("=" * 80)

results = retrieve_with_similarity(q5["question"], k=10)

for rank, (doc, score) in enumerate(results, start=1):

    print(f"\nRank {rank}")
    print("Score:", round(score, 4))
    print("Document:", doc.metadata.get("document_name"))
    print("Page:", doc.metadata.get("page_number"))
    print("Pages:", doc.metadata.get("page_numbers"))
    print("Section:", doc.metadata.get("section"))
    print("Text:")
    print(doc.page_content[:1000])
    print("-" * 80)

Question: What is usually the first medicine given for type 2 diabetes with no other health problems?
Expected guideline: ['type-2']
Expected page: 32

Rank 1
Score: 0.6436
Document: Type-2 diabetes.pdf
Page: 71
Pages: [71]
Section: 1.21 Preventing diabetic ketoacidosis when taking
Text:
effects and intolerances from the first medicine to be identified before the second is
introduced. In line with current practice, the committee recommended starting with
metformin and then adding the SGLT-2 inhibitor without delay once metformin
--------------------------------------------------------------------------------

Rank 2
Score: 0.6362
Document: Type-2 diabetes.pdf
Page: 129
Pages: [129]
Section: 1.45 Antiplatelet therapy
Text:
Update information
February 2026: We reviewed evidence on medicines for type 2 diabetes, for people with
no relevant comorbidities as well as for people with common comorbidities.
We made new and updated recommendations on metformin, SGLT-2 inhibitors, GLP-1
---------

In [26]:
# Inspect Page 32 directly

for doc in chunks:
    pages = doc.metadata.get("page_numbers", [])

    if isinstance(pages, str):
        pages = _extract_pages(
            pages,
            doc.metadata.get("page_number")
        )

    if 32 in pages:
        print("Document:", doc.metadata.get("document_name"))
        print("Page:", doc.metadata.get("page_number"))
        print("Pages:", pages)
        print("Section:", doc.metadata.get("section"))
        print("\nTEXT:")
        print(doc.page_content)
        print("=" * 100)

Document: Type-1 diabetes.pdf
Page: 31
Pages: [31, 32]
Section: 1.9 Hypoglycaemia awareness and management

TEXT:
control possible. [2004]
1.9.13 Make hypoglycaemia advice available to all adults with type 1 diabetes, to help
them find the best possible balance with any insulin regimen. (See the sections
on insulin therapy and insulin injection delivery.) [2004]
Document: Type-1 diabetes.pdf
Page: 32
Pages: [32]
Section: 1.9 Hypoglycaemia awareness and management

TEXT:
on insulin therapy and insulin injection delivery.) [2004]
1.9.14 If hypoglycaemia becomes unusually problematic or increases in frequency,
review the following possible causes:
• inappropriate insulin regimens (incorrect dose distributions and insulin types)
Document: Type-1 diabetes.pdf
Page: 32
Pages: [32]
Section: 1.9 Hypoglycaemia awareness and management

TEXT:
• meal and activity patterns, including alcohol
• injection technique and skills, including insulin resuspension if necessary
• injection site problems
• p